> ## SOLUTION / ANSWER KEY &mdash; Lab 10.7
> This is the **completed** notebook (all `___` blanks filled). For the student version, open
> [`../lab-07-the-eval-set.ipynb`](../lab-07-the-eval-set.ipynb). Trainer use &mdash; or self-check after you've tried it yourself.

# Lab 10.7 &mdash; Build an Eval Set

**Level:** Intermediate &nbsp;|&nbsp; **Est. time:** 30 min &nbsp;|&nbsp; **Day 5 &middot; Module 10 &mdash; Ethics &amp; Responsible AI in Agentic Systems**

### What you'll do
- Define cases with inputs and expected outputs
- Run an agent over the set and compute a pass-rate
- See why one passing run is not evidence of quality

> **How this lab works (near-real):** read the **Concept**, fill the real `___` blanks in **Build it**, then **run it &amp; observe**. The responsible-AI logic here &mdash; injection defence, least privilege, trace-reading, fairness, the eval loop, the guardrails &mdash; is **real, deterministic Python** you can run offline. The **Advanced agent labs (10&ndash;12)** additionally drive a **real Groq model** through `create_agent`: **Run it for real** and **read the trace**. Finish with an open **Your turn**. There is **no auto-grader**; the goal is a responsible agent and a trace you can read.

> **Framework note:** these labs use the **real** LangChain 1.x (`langchain`, `langchain-core`, `langchain-groq`). The agent labs use a **real hosted model** (`ChatGroq("openai/gpt-oss-20b")` &mdash; reliable tool-calling via `create_agent`); the guardrail/eval/trace logic is genuine rule-based Python. If `GROQ_API_KEY` is unset, the run cells print how to set it instead of crashing. A `@tool` must **catch its own errors and return a string** &mdash; a tool that *raises* can abort the whole agent run. This is the **course finale** &mdash; Lab 5.2: responsible-AI frameworks &amp; **debugging agent errors**.

**Reference:** [Module 10 slides &mdash; The eval loop](../../presentation/day5-module10-ethics-responsible-ai.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 10 labs](./index.html)

In [ ]:
# Setup -- run me first
import os, time, pathlib
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True), override=True)   # GROQ_API_KEY (+ other keys)

WORK = os.path.join(os.environ.get("TEMP") or os.environ.get("TMP") or "/tmp", "biaa-lab-10-07")
os.makedirs(WORK, exist_ok=True)

def groq_ready():
    """True if a GROQ_API_KEY is configured -- the 'Run it for real' cells self-skip if not."""
    return bool(os.environ.get("GROQ_API_KEY"))

from langchain_groq import ChatGroq
# Day-5 provider: a REAL hosted model with reliable tool-calling via create_agent.
MODEL = "openai/gpt-oss-20b"
llm = ChatGroq(model=MODEL, temperature=0) if groq_ready() else None

def with_backoff(fn, tries=5):
    """Run fn(); retry with backoff on Groq rate limits (HTTP 429). Other errors propagate."""
    last = None
    for attempt in range(tries):
        try:
            return fn()
        except Exception as e:
            last = e
            if "429" in str(e) or "rate limit" in str(e).lower() or "rate_limit" in str(e).lower():
                wait = 5 * (attempt + 1)
                print(f"(rate-limited -- retrying in {wait}s)")
                time.sleep(wait)
                continue
            raise
    raise last

def print_trace(result):
    """Print a REAL agent message trace: tool calls the model made, tool observations, final answer."""
    for m in result["messages"]:
        for tc in (getattr(m, "tool_calls", None) or []):
            print("TOOL CALL:", tc["name"], tc["args"])
        if type(m).__name__ == "ToolMessage":
            print("OBS:", str(m.content)[:200])
        elif str(getattr(m, "content", "")).strip():
            print(type(m).__name__, ":", str(m.content)[:300])

if groq_ready():
    print("Groq ready | model:", MODEL, "| WORK:", WORK)
else:
    print("GROQ_API_KEY not set -- add it to the repo .env (free key at console.groq.com).")
    print("(The 'Run it for real' cells will print this note instead of crashing.)  WORK:", WORK)

## Concept
Agent quality is fuzzy and non-deterministic, so *&ldquo;it worked once&rdquo;* is an illusion (deck slide
17). The fix is to **measure**: build an **eval set** &mdash; representative inputs with expected behaviour,
including the failures you've found &mdash; and compute a **pass-rate**. Then you iterate against a target,
not a vibe. (You'll **reuse this exact `run_eval`** to score the capstone agent in Lab 12.)

In [ ]:
# A tiny agent under test (deterministic): classifies a query's intent.
def agent_fn(text):
    t = text.lower()
    if "refund" in t: return "billing"
    if "crash" in t: return "tech"
    return "general"
print("agent ready:", agent_fn("I need a refund"))

## Build it
Complete `run_eval`: count how many cases the agent gets right and compute the rate. This is the harness
you'll point at a *real* agent in Lab 12.

In [ ]:
def run_eval(fn, cases):
    # cases: list of {input, expected}
    passed = sum(1 for c in cases if fn(c["input"]) == c["expected"])
    return {"passed": passed, "total": len(cases), "rate": passed / len(cases)}

CASES = [
    {"input": "I want a refund", "expected": "billing"},
    {"input": "the app keeps crashing", "expected": "tech"},
    {"input": "what are your hours", "expected": "general"},
]

try:
    print("eval         :", run_eval(agent_fn, CASES))
    print("a broken agent:", run_eval(lambda t: "billing", CASES))
except Exception as e:
    print("(Fill the ___ blanks above, then re-run.)", type(e).__name__)

## What to notice
- The good `agent_fn` scores **1.0**; a "just say billing" agent scores below 1.0 &mdash; the metric distinguishes them.
- `run_eval` takes **any** `fn` &mdash; a deterministic function here, a real Groq agent's decision in Lab 12. Same harness.
- One passing run is anecdote; a pass-rate over a set is **evidence**. That's the whole point of the eval loop.

## Your turn (open task &mdash; no grader)
Add two adversarial cases to `CASES` &mdash; an ambiguous query and one that should escalate &mdash; and
re-run. **What good looks like:** the pass-rate drops to reflect the new hard cases, giving you a real target
to improve against instead of a vibe. Keep the ones your agent fails; they're your regression set.

In [ ]:
# --- Reference answer (ONE good way to do the 'Your turn' task -- compare with your own) ---
CASES2 = CASES + [
    {"input": "I love your product",                 "expected": "escalate"},  # ambiguous: agent says 'general'
    {"input": "the app crashed after you charged me", "expected": "billing"},  # two intents: agent sees 'crash'
]
print("expanded eval:", run_eval(agent_fn, CASES2))   # rate drops -- these two are now your regression set

---
### Key takeaway
An eval set with a pass-rate turns 'it worked once' into a measurable target you can improve against. It's the engine of the improve step -- and, as the next lab shows, your safety net.

[&#8592; All Module 10 labs](./index.html) &nbsp;&middot;&nbsp; [Module 10 slides](../../presentation/day5-module10-ethics-responsible-ai.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>